In [1]:
import math

class Value:
    def __init__(self,data,_children=(),_op='',label=''):
        self.data= data
        self.grad=0.0
        self._backward= lambda:None
        self.label= label
        self._prev= set(_children)
        self._op=_op
    
    def __repr__(self):
        return f"Value(data={self.data})"
    
    def __add__(self,other):
        other= other if isinstance(other,Value) else Value(other)
        out= Value(self.data + other.data, (self,other),'+')
        
        def _backward():
            self.grad+= out.grad
            other.grad+= out.grad
        
        out._backward= _backward
        return out
    
    def __mul__(self, other):
        other= other if isinstance(other,Value) else Value(other)
        out= Value(self.data * other.data, (self,other),'*')
        
        def _backward():
            self.grad+= other.data*out.grad
            other.grad+= self.data* out.grad
        
        out._backward= _backward
        return out
    
    def __pow__(self, other):
        assert isinstance(other,(int,float))
        out = Value(self.data**other, (self,),f'**{other}')
        
        def _backward():
            self.grad= (other* (self.data**(other-1))) * out.grad
        
        out._backward= _backward
        return out
    
    def __rmul__(self,other):
        return self*other
    
    def __radd__(self, other):
        return self+other
    
    def __truediv__(self, other):
        return self* other**-1
    
    def __neg__(self):
        return self*-1
    
    def __sub__(self, other):
        return self + (-other)
    
    def tanh(self):
        x= self.data
        t= (math.exp(2*x)-1)/(math.exp(2*x)+1)
        out= Value(t, (self,),'tanh')
        
        def _backward():
            self.grad+= (1- t**2)* out.grad
        
        out._backward= _backward
        return out
    
    def exp(self):
        x= self.data
        e= (math.exp(x))
        out= Value(e, (self,),'exp')
        
        def _backward():
            self.grad+= e * out.grad
        
        out._backward= _backward
        return out
    

In [2]:
a= Value(2.0,label='a')
b= Value(-3.0,label='b')
c= Value(10.0,label='c')

d= a*b + c
d.label= 'd'
e= a*b
e.label= 'e'
f= Value(-2.0,label='f')
L= d*f
L.label= 'L'

In [3]:
a/b

Value(data=-0.6666666666666666)

In [4]:
import random

class Neuron:
    def __init__(self,nin):
        self.w= [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b= Value(random.uniform(-1,1))
    
    def __call__(self, x):
        act= sum((wi*xi for wi,xi in zip(self.w,x)),self.b)
        out= act.tanh()
        return out
    
    def parameters(self):
        params= self.w + [self.b]
        return params

class Layer:
    def __init__(self,nin,nout):
        self.neurons= [Neuron(nin) for _ in range(nout)]
    
    def __call__(self, x):
        outs= [n(x) for n in self.neurons]
        return outs[0] if len(outs)==1 else outs
    
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]
    
class MLP:
    def __init__(self,nin,nouts):
        sz= [nin] + nouts
        self.layers= [Layer(sz[i],sz[i+1]) for i in range(len(nouts))]
    
    def __call__(self,x):
        for layer in self.layers:
            x= layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


In [5]:
x= [2.0,3.0,-1.0]
n= MLP(3,[4,4,1])
n(x)

Value(data=0.39550396664705445)

In [6]:
xs= [
        [2.0,3.0,-1.0],
        [3.0,-1.0,2.0],
        [4.5,1.4,5.6],
        [-2.0,3.0,4.0]
    ]
ys= [0.5,-0.5,0.0,1.0]

In [7]:
for i in range(20):
    
    ypred= [n(x) for x in xs]
    loss= sum((Y-y)**2 for Y,y in zip(ypred,ys))
    
    for p in n.parameters():
        p.grad=0
    loss._backward()
    
    for p in n.parameters():
        p.data+= -0.05*p.grad
    
    print(i, loss)



0 Value(data=1.0484244071163527)
1 Value(data=1.0484244071163527)
2 Value(data=1.0484244071163527)
3 Value(data=1.0484244071163527)
4 Value(data=1.0484244071163527)
5 Value(data=1.0484244071163527)
6 Value(data=1.0484244071163527)
7 Value(data=1.0484244071163527)
8 Value(data=1.0484244071163527)
9 Value(data=1.0484244071163527)
10 Value(data=1.0484244071163527)
11 Value(data=1.0484244071163527)
12 Value(data=1.0484244071163527)
13 Value(data=1.0484244071163527)
14 Value(data=1.0484244071163527)
15 Value(data=1.0484244071163527)
16 Value(data=1.0484244071163527)
17 Value(data=1.0484244071163527)
18 Value(data=1.0484244071163527)
19 Value(data=1.0484244071163527)
